# 8 — Shared embedding denoising experiments

Can we expose the raw semantic signal by removing a small amount of dominant variation,
then project both words into one 2-D space? This notebook uses **only the labelled sample**
and preserves **century centroids**. Run from `scripts/` in `phylosen_venv`.

There are no cross-word sense anchors, merged inventories, LDA axes, or learned semantic
alignment. Sense labels are used for splitting and evaluation, not to fit the filters
or display projections. All occurrences receive the same fitted transformation.

**Coordinates versus directions:** zeroing original embedding coordinates depends on the
encoder's arbitrary basis. Removing a principal component removes a direction that can
involve every coordinate. Existing top-k dimension-selection failures do not establish
that removing a few nuisance directions must fail too.

A large component is only a *candidate* nuisance: it may contain meaning, language,
register, author, or dating information. The experiment measures what is lost.

In [ ]:
import importlib
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.linalg import orthogonal_procrustes
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
from threadpoolctl import threadpool_limits
from IPython.display import display

import embedding_denoising
importlib.reload(embedding_denoising)
from embedding_denoising import (FilterSpec, SharedFilterBank, century_centroids,
                                 cross_word_distances)

SEED = 42
MODEL = "qwen3.8:27b"  # explicit LLM sample label column; no embedding predictions
LABEL_COL = f"class_{MODEL}"
TERMS = ["passio", "pathos"]
DATA = Path("../data/large_files")
OUT = Path("../figures/embedding-denoising")
OUT.mkdir(parents=True, exist_ok=True)
INPUT_SPECS = {
    "philberta": ["emb_philberta_l11"],
    "mdeberta": ["emb_mdeberta_l9"],
    "concat": ["emb_philberta_l11", "emb_mdeberta_l9"],
}
SELECTED_INPUT = "concat"  # inspect either encoder independently too
TEST_FRACTION = .25
GROUP_BY_WORK = True       # prevent occurrences of the same work crossing the split
MIN_CENTURY = 5  # minimum usages per temporal bin (also for 200-year bins)
K_NEIGHBORS = 10
RUN_UMAP = False           # optional nonlinear comparison, trained without sense labels
RUN_MLP = True             # slower, stronger check of retained sense signal
MLP_FILTERS = ["raw", "center_l2", "drop_pc_2", "drop_pc_5"]
MLP_INPUT = SELECTED_INPUT
THREADS = 2
random.seed(SEED)
np.random.seed(SEED)

## Sample, dates, and one fixed train/test split

Use the current LLM sense inventories, omit uncertain sense 0, and derive dates only
when absent. The date draw is seeded once and shared by every experiment. Missing dates
exclude a row from temporal plots only. Hold out entire `grela_id` works when available;
missing work IDs receive unique row groups. Word/sense coverage is checked before fitting.
The holdout is exploratory: repeated comparisons can overfit it, so final selection needs
another split or nested cross-validation.

In [ ]:
import tempun

def century_midpoint(year):
    if pd.isna(year) or not np.isfinite(float(year)):
        return np.nan
    year = int(year)
    if year == 0:
        return np.nan
    return ((year - 1) // 100 if year > 0 else year // 100) * 100 + 50

def draw_date(row):
    value = tempun.model_date(row["not_before"], row["not_after"])
    while isinstance(value, (list, tuple, np.ndarray)):
        if len(value) == 0:
            return np.nan
        value = value[0]
    return value

frames, SENSE_NAMES = [], {}
for term in TERMS:
    df = pd.read_parquet(DATA / f"{term}_sample2000_classified_embeddings.parquet")
    if LABEL_COL not in df:
        raise KeyError(f"{term}: missing requested LLM labels {LABEL_COL}")
    labels = pd.to_numeric(df[LABEL_COL], errors="coerce")
    df = df.loc[labels.notna() & (labels != 0)].copy()
    df["source_row"] = df.index
    df["sense"] = labels.loc[df.index].astype(int)
    df["word"] = term
    if "random_date" not in df:
        df["random_date"] = df.apply(draw_date, axis=1)
    df["century"] = df["random_date"].map(century_midpoint)
    module = importlib.import_module(f"llmprompt_{term}")
    SENSE_NAMES[term] = {int(s): name.replace("_", " ")
                        for s, name in importlib.reload(module).LABEL_MAP.items() if s != 0}
    frames.append(df)
DF = pd.concat(frames, ignore_index=True)
X_INPUTS = {name: np.concatenate([np.stack(DF[col]) for col in cols], axis=1).astype(float)
            for name, cols in INPUT_SPECS.items()}
finite = np.logical_and.reduce([np.isfinite(X).all(axis=1) for X in X_INPUTS.values()])
if not finite.all():
    print(f"Dropping {int((~finite).sum())} rows with nonfinite embeddings across inputs.")
    DF = DF.loc[finite].reset_index(drop=True)
    X_INPUTS = {name: X[finite] for name, X in X_INPUTS.items()}
WORDS = DF.word.to_numpy()
Y = DF.sense.to_numpy()
JOINT_LABELS = DF.word + ":" + DF.sense.astype(str)
ALL_CLASSES = set(JOINT_LABELS)

if GROUP_BY_WORK:
    if "grela_id" not in DF:
        raise ValueError("No work IDs: explicitly set GROUP_BY_WORK=False for a row split.")
    GROUPS = np.array([f"work:{v}" if pd.notna(v) else f"missing-row:{i}"
                       for i, v in enumerate(DF.grela_id)])
    splitter = GroupShuffleSplit(n_splits=100, test_size=TEST_FRACTION, random_state=SEED)
    for TRAIN, TEST in splitter.split(DF, groups=GROUPS):
        if set(JOINT_LABELS.iloc[TRAIN]) == ALL_CLASSES and set(JOINT_LABELS.iloc[TEST]) == ALL_CLASSES:
            break
    else:
        raise ValueError("Cannot cover all senses with a work split. Inspect rare senses before changing the split.")
    assert not set(GROUPS[TRAIN]) & set(GROUPS[TEST])
else:
    TRAIN, TEST = train_test_split(np.arange(len(DF)), test_size=TEST_FRACTION,
                                   stratify=JOINT_LABELS, random_state=SEED)
print("split:", "held-out works" if GROUP_BY_WORK else "held-out rows")
display(pd.crosstab([DF.word, DF.sense], np.where(np.isin(np.arange(len(DF)), TRAIN), "train", "test")))
print("embedding shapes:", {name: X.shape for name, X in X_INPUTS.items()})
# Preserve exact split membership and dating draw alongside the exported results.
manifest = DF[[c for c in ["word", "source_row", "sense", "grela_id", "random_date", "century"] if c in DF]].copy()
manifest["split"] = np.where(np.isin(np.arange(len(DF)), TRAIN), "train", "test")
manifest.to_csv(OUT / "sample-split-and-dates.csv", index=False)

## Filtering sweep

1. **Raw:** unchanged embeddings; PCA will supply its own common centering.
2. **Centered + unit length:** a control for changing cosine geometry through centering.
3. **Remove top 1, 2, 5, 10, 20 PCs:** an adaptation of
   [All-but-the-Top](https://openreview.net/pdf?id=HkuGJ3kCb). That paper studies word
   representations; benefit for these contextual embeddings is a hypothesis.
4. **Half-strength removal of top 5 PCs:** avoid an all-or-nothing decision.
5. **Zero top 10 or 50 high-variance original coordinates:** test literal dimension removal.
6. **Optional word-mean direction attenuation:** a confounding sensitivity control, off
   by default. Word identity is not automatically nuisance, and removing its direction
   can remove real semantic differences.

All filtered variants share the same training-fitted origin, equal word weights and
unit-length convention. Compare them against `center_l2`, as well as `raw`, to distinguish
filtering from normalization. No separate centering by word is applied at transform time.

In [ ]:
FILTERS = [FilterSpec("raw"), FilterSpec("center_l2", "center", normalize=True)]
FILTERS += [FilterSpec(f"drop_pc_{k}", "pc", count=k, normalize=True)
            for k in [1, 2, 5, 10, 20]]
FILTERS += [FilterSpec("half_pc_5", "pc", count=5, strength=.5, normalize=True)]
FILTERS += [FilterSpec(f"drop_coordinate_{k}", "coordinate", count=k, normalize=True)
            for k in [10, 50]]
INCLUDE_WORD_DIRECTION = False
if INCLUDE_WORD_DIRECTION:
    FILTERS += [FilterSpec(f"word_direction_{a}", "word_direction", strength=a, normalize=True)
                for a in [.5, 1.0]]

BANKS, FILTERED = {}, {}
with threadpool_limits(limits=THREADS):
    for name, X in X_INPUTS.items():
        bank = SharedFilterBank(max_components=64, seed=SEED).fit(X[TRAIN], WORDS[TRAIN])
        BANKS[name] = bank
        FILTERED[name] = {spec.name: bank.transform(X, spec) for spec in FILTERS}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, bank in BANKS.items():
    axes[0].plot(np.arange(1, len(bank.explained_variance_ratio_) + 1), bank.explained_variance_ratio_, label=name)
    axes[1].plot(np.arange(1, len(bank.explained_variance_ratio_) + 1), np.cumsum(bank.explained_variance_ratio_), label=name)
axes[0].set(xlabel="Principal component", ylabel="Fraction of training variance")
axes[1].set(xlabel="Number of leading PCs", ylabel="Cumulative fraction of training variance")
for ax in axes:
    ax.legend(); ax.grid(alpha=.2)
fig.tight_layout()
plt.show()

## Shared 2-D PCA and held-out diagnostics

Fit each display PCA on **training rows only**, with equal total word weight. Rotate the
2-D result orthogonally toward the raw display for visual comparison; this uses training
coordinates only and does not change distances or impose sense correspondences.

Measure sense macro-F1 within each word, before and after 2-D compression; word identity
balanced accuracy is a diagnostic, not a minimization target. Neighbour overlap measures
how much the filter changes raw cosine neighbourhoods. High overlap is not proof of
semantic quality; low overlap is not proof of improvement.

In [ ]:
def display_pca(X):
    # Weighted covariance via the same shared PCA bank; no labels enter this fit.
    fitted = SharedFilterBank(max_components=2, seed=SEED).fit(X[TRAIN], WORDS[TRAIN])
    return (X - fitted.mean_) @ fitted.components_[:2].T

def heldout_knn(X, term, metric):
    train = TRAIN[WORDS[TRAIN] == term]
    test = TEST[WORDS[TEST] == term]
    model = KNeighborsClassifier(n_neighbors=min(K_NEIGHBORS, len(train)), metric=metric)
    model.fit(X[train], Y[train])
    pred = model.predict(X[test])
    return f1_score(Y[test], pred, average="macro", zero_division=0)

def neighbor_ids(X, query):
    model = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric="cosine").fit(X[TRAIN])
    return model.kneighbors(X[query], return_distance=False)

# A fixed query subset keeps the neighbourhood comparison inexpensive.
QUERY = np.random.default_rng(SEED).choice(TEST, min(150, len(TEST)), replace=False)
COORDS, diagnostic_rows = {}, []
with threadpool_limits(limits=THREADS):
    for name, variants in FILTERED.items():
        COORDS[name] = {}
        reference = display_pca(variants["raw"])
        reference_neighbors = neighbor_ids(variants["raw"], QUERY)
        for spec in FILTERS:
            X = variants[spec.name]
            Z = display_pca(X)
            rotation, _ = orthogonal_procrustes(Z[TRAIN], reference[TRAIN])
            Z = Z @ rotation
            COORDS[name][spec.name] = Z
            probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
            probe.fit(X[TRAIN], WORDS[TRAIN])
            word_score = balanced_accuracy_score(WORDS[TEST], probe.predict(X[TEST]))
            neighbors = neighbor_ids(X, QUERY)
            overlap = np.mean([len(set(a) & set(b)) / K_NEIGHBORS
                               for a, b in zip(reference_neighbors, neighbors)])
            removed = (BANKS[name].explained_variance_ratio_[:spec.count].sum()
                       if spec.kind == "pc" else np.nan)
            for term in TERMS:
                diagnostic_rows.append(dict(input=name, filter=spec.name, word=term,
                    sense_f1_hd=heldout_knn(X, term, "cosine"),
                    sense_f1_2d=heldout_knn(Z, term, "euclidean"),
                    word_bal_acc=word_score, raw_neighbor_overlap=overlap,
                    pc_variance_targeted=removed))
DIAGNOSTICS = pd.DataFrame(diagnostic_rows)
DIAGNOSTICS.to_csv(OUT / "heldout-diagnostics.csv", index=False)
display(DIAGNOSTICS.round(3))

## MLP signal-retention check

Use the same `(512, 128)` ReLU architecture and scaling convention as notebook 5, refitted
inside this notebook's training partition. Compare raw embeddings with selected filters
on identical held-out works. These scores are not directly comparable with notebook 5's
row-level cross-validation. Increase `MLP_FILTERS` after inspecting the first sweep.
This is a diagnostic of remaining signal, not a projection objective.

In [ ]:
MLP_RESULTS = pd.DataFrame()
if RUN_MLP:
    rows = []
    with threadpool_limits(limits=THREADS):
        for filter_name in MLP_FILTERS:
            X = FILTERED[MLP_INPUT][filter_name]
            for term in TERMS:
                train = TRAIN[WORDS[TRAIN] == term]
                test = TEST[WORDS[TEST] == term]
                pipeline = make_pipeline(StandardScaler(), MLPClassifier(
                    hidden_layer_sizes=(512, 128), activation="relu", solver="adam",
                    alpha=1e-4, max_iter=500, random_state=SEED))
                pipeline.fit(X[train], Y[train])
                pred = pipeline.predict(X[test])
                rows.append(dict(input=MLP_INPUT, filter=filter_name, word=term,
                    accuracy=accuracy_score(Y[test], pred),
                    f1_macro=f1_score(Y[test], pred, average="macro", zero_division=0),
                    iterations=pipeline[-1].n_iter_,
                    reached_max_iter=pipeline[-1].n_iter_ >= pipeline[-1].max_iter))
                print(rows[-1], flush=True)
    MLP_RESULTS = pd.DataFrame(rows)
    MLP_RESULTS.to_csv(OUT / "heldout-mlp.csv", index=False)
    display(MLP_RESULTS.round(3))
else:
    print("MLP check disabled; set RUN_MLP=True to evaluate retained nonlinear signal.")

## Shared maps and century trajectories

Colours are assigned consistently per `(word, sense)` across all panels; square = passio,
diamond = pathos. Set `TEMPORAL_BIN_YEARS` in the plotting cell below: **200** for bi-centuries or **100**
for centuries. CE 1–200, 201–400, … are plotted at 100, 300, …; BCE −200–−1 is
plotted at −100. `MIN_CENTURY` applies to the total usages per word/sense/bin.
Every marker averages **projected individual usages**, weighted equally across the bin.
Lines connect adjacent valid bin midpoints only; no smoothing or bridging gaps. The figures
use all sample rows, so they are descriptive; prediction diagnostics above use the holdout.
Axes are independently fitted and rotated for comparison, so numerical scales across
panels need not agree.

In [ ]:
# Change the width here and rerun this cell; no model retraining is needed.
TEMPORAL_BIN_YEARS = 200  # 100: centuries; 200: bi-centuries
importlib.reload(embedding_denoising)
century_centroids = embedding_denoising.century_centroids

PALETTE = ["#3B6FB6", "#D48722", "#30865D", "#B74444", "#8A5DA8",
           "#269FB3", "#D35F9F", "#73513D", "#617E3E", "#34495E"]
KEYS = [(t, int(s)) for t in TERMS for s in sorted(DF.loc[DF.word == t, "sense"].unique())]
COLORS = {key: PALETTE[i % len(PALETTE)] for i, key in enumerate(KEYS)}
MARKERS = {"passio": "s", "pathos": "D"}
PLOT_FILTERS = ["raw", "center_l2", "drop_pc_1", "drop_pc_2", "drop_pc_5", "drop_coordinate_50"]
CENTURIES = {}

def plot_comparison(coords, prefix):
    fig_map, map_axes = plt.subplots(2, 3, figsize=(15, 9))
    fig_time = plt.figure(figsize=(15, 12))
    legend = [Line2D([0], [0], color=COLORS[(t, s)], marker=MARKERS[t], linestyle="",
              label=f"{t}: {SENSE_NAMES[t].get(s, str(s))}") for t, s in KEYS]
    for i, name in enumerate(coords):
        Z = coords[name]
        ax = map_axes.flat[i]
        for t, s in KEYS:
            mask = (WORDS == t) & (Y == s)
            ax.scatter(Z[mask, 0], Z[mask, 1], color=COLORS[(t, s)], marker=MARKERS[t],
                       s=9, alpha=.25)
        ax.set(title=name, xlabel="Shared component 1", ylabel="Shared component 2")
        points = century_centroids(DF, Z, MIN_CENTURY, bin_years=TEMPORAL_BIN_YEARS)
        CENTURIES[(prefix, name)] = points
        points.to_csv(OUT / f"{prefix}-{name}-{TEMPORAL_BIN_YEARS}yr-centroids.csv", index=False)
        at = fig_time.add_subplot(2, 3, i + 1, projection="3d")
        for t, s in KEYS:
            g = points[(points.word == t) & (points.sense == s) & points.valid].sort_values("year")
            if g.empty:
                continue
            for run in np.split(np.arange(len(g)), np.flatnonzero(np.diff(g.year) != TEMPORAL_BIN_YEARS) + 1):
                r = g.iloc[run]
                if len(r) > 1:
                    at.plot(r.x, r.y, r.year, color=COLORS[(t, s)], alpha=.85, lw=1.2)
            at.scatter(g.x, g.y, g.year, color=COLORS[(t, s)], marker=MARKERS[t],
                       s=22, alpha=.9, depthshade=False)
        at.set(title=name, xlabel="Component 1", ylabel="Component 2", zlabel=f"Year ({TEMPORAL_BIN_YEARS}-year midpoint)")
        at.view_init(elev=25, azim=-60)
        at.set_proj_type("ortho")
        at.set_box_aspect((1, 1, 1.4))
    for ax in map_axes.flat[len(coords):]:
        ax.set_visible(False)
    for fig, suffix in [(fig_map, "maps"), (fig_time, "centuries")]:
        title = f"{prefix}: sample, shared label-free projection"
        if fig is fig_time:
            title += f"; {TEMPORAL_BIN_YEARS}-year centroids (n ≥ {MIN_CENTURY})"
        fig.suptitle(title, fontsize=14)
        fig.legend(handles=legend, loc="lower center", ncol=2, frameon=False, fontsize=8)
        fig.tight_layout(rect=(0, .15, 1, .96))
        if fig is fig_time:
            fig.subplots_adjust(left=.03, right=.97, bottom=.19, top=.93, hspace=.38, wspace=.18)
        for ext in ["png", "pdf"]:
            filename = (f"{prefix}-{TEMPORAL_BIN_YEARS}yr-centroids" if fig is fig_time
                        else f"{prefix}-{suffix}")
            fig.savefig(OUT / f"{filename}.{ext}", dpi=180, bbox_inches="tight")
    plt.show()

plot_comparison({name: COORDS[SELECTED_INPUT][name] for name in PLOT_FILTERS},
                prefix=f"{SELECTED_INPUT}-pca")

## Focused figure — drop_pc_1

A standalone view of the same `drop_pc_1` coordinates used in the comparison above.
It reuses `TEMPORAL_BIN_YEARS` and `MIN_CENTURY`; rerun this cell after changing either
setting. `FOCUS_ELEV` and `FOCUS_AZIM` control the viewing angle. `FOCUS_XLIM` and `FOCUS_YLIM` optionally set explicit
display-axis bounds; by default, bounds tightly surround the visible centroids.
No refitting is needed.


In [ ]:
FOCUS_FILTER = "drop_pc_1"
FOCUS_ELEV, FOCUS_AZIM = 25, -60
FOCUS_XLIM = None  # optional explicit pair, e.g. (-0.15, 0.25)
FOCUS_YLIM = None  # optional explicit pair, e.g. (-0.20, 0.30)
FOCUS_AXIS_PADDING = .02  # automatic limits: 2% beyond the visible centroids

PALETTE = ["#3B6FB6", "#D48722", "#30865D", "#B74444", "#8A5DA8",
           "#269FB3", "#D35F9F", "#73513D", "#617E3E", "#34495E"]
COLORS = {key: PALETTE[i % len(PALETTE)] for i, key in enumerate(KEYS)}
MARKERS = {"passio": "s", "pathos": "D"}

FOCUS_CENTROIDS = century_centroids(
    DF, COORDS[SELECTED_INPUT][FOCUS_FILTER], MIN_CENTURY,
    bin_years=TEMPORAL_BIN_YEARS)
fig_focus = plt.figure(figsize=(11.5, 9.2))
focus_grid = fig_focus.add_gridspec(1, 2, width_ratios=[3.1, 1.4])
ax_focus = fig_focus.add_subplot(focus_grid[0, 0], projection="3d")
ax_focus_legend = fig_focus.add_subplot(focus_grid[0, 1])
ax_focus_legend.set_axis_off()
focus_handles = {t: [] for t in TERMS}
for t, s in KEYS:
    g = FOCUS_CENTROIDS[(FOCUS_CENTROIDS.word == t)
        & (FOCUS_CENTROIDS.sense == s) & FOCUS_CENTROIDS.valid].sort_values("year")
    if g.empty:
        continue
    color, marker = COLORS[(t, s)], MARKERS[t]
    runs = np.split(np.arange(len(g)),
                    np.flatnonzero(np.diff(g.year) != TEMPORAL_BIN_YEARS) + 1)
    for run in runs:
        r = g.iloc[run]
        if len(r) > 1:
            ax_focus.plot(r.x, r.y, r.year, color=color, lw=2, alpha=.85)
    ax_focus.scatter(g.x, g.y, g.year, color=color, marker=marker,
                     s=45, alpha=.95, depthshade=False, edgecolors="white", linewidths=.5)
    focus_handles[t].append(Line2D([0], [0], color=color, marker=marker,
        linewidth=2, markersize=7, label=SENSE_NAMES[t].get(s, str(s)), markeredgecolor="white", markeredgewidth=.5))
visible = FOCUS_CENTROIDS.loc[FOCUS_CENTROIDS.valid]
for column, override, set_limit in [("x", FOCUS_XLIM, ax_focus.set_xlim),
                                    ("y", FOCUS_YLIM, ax_focus.set_ylim)]:
    if override is not None:
        set_limit(*override)
    elif not visible.empty:
        lo, hi = visible[column].min(), visible[column].max()
        pad = max((hi - lo) * FOCUS_AXIS_PADDING, 1e-6)
        set_limit(lo - pad, hi + pad)
ax_focus.set_xlabel("Shared display component 1", labelpad=10)
ax_focus.set_ylabel("Shared display component 2", labelpad=10)
ax_focus.set_zlabel(f"Year ({TEMPORAL_BIN_YEARS}-year midpoint)", labelpad=10)
ax_focus.view_init(elev=FOCUS_ELEV, azim=FOCUS_AZIM)
ax_focus.set_proj_type("ortho")
ax_focus.set_box_aspect((1, 1, 1.45))
for axis in [ax_focus.xaxis, ax_focus.yaxis, ax_focus.zaxis]:
    axis.set_pane_color((.98, .98, .98, .25))
    axis._axinfo["grid"]["color"] = (.7, .7, .7, .3)
for term, legend_y in [("passio", .66), ("pathos", .40)]:
    legend = ax_focus_legend.legend(handles=focus_handles[term], loc="center left",
        bbox_to_anchor=(0, legend_y), frameon=False, fontsize=9,
        title=term, title_fontsize=11, handlelength=1.7,
        handletextpad=.6, labelspacing=.65, borderaxespad=0)
    legend.get_title().set_fontweight("bold")
    ax_focus_legend.add_artist(legend)
# Draw first so tight_layout can measure projected 3-D labels.
fig_focus.canvas.draw()
fig_focus.tight_layout(pad=1.2, w_pad=.3)
focus_output = OUT / f"{SELECTED_INPUT}-pca-{FOCUS_FILTER}-{TEMPORAL_BIN_YEARS}yr-centroids"
FOCUS_CENTROIDS.to_csv(focus_output.with_suffix(".csv"), index=False)
fig_focus.savefig(focus_output.with_suffix(".png"), dpi=300, bbox_inches="tight", pad_inches=.1)
fig_focus.savefig(focus_output.with_suffix(".pdf"), bbox_inches="tight", pad_inches=.1)
plt.show()


## Cross-word distances before display compression

Inspect the complete sense-by-sense table, rather than selecting only similarly named
senses. These cosine distances are descriptive sample estimates, not semantic ground
truth. Centering changes cosine geometry; contrast filtered variants with `center_l2`.
No table is used to tune a filter toward desired cross-word overlap.

In [ ]:
DISTANCE_TABLES = {}
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, name in zip(axes, ["raw", "center_l2", "drop_pc_5"]):
    table = cross_word_distances(FILTERED[SELECTED_INPUT][name], DF, SENSE_NAMES)
    DISTANCE_TABLES[name] = table
    table.to_csv(OUT / f"{SELECTED_INPUT}-{name}-cross-word-distances.csv")
    img = ax.imshow(table.to_numpy(), vmin=0, vmax=2, cmap="viridis")
    ax.set_xticks(np.arange(len(table.columns)), table.columns, rotation=65, ha="right", fontsize=7)
    ax.set_yticks(np.arange(len(table.index)), table.index, fontsize=7)
    ax.set_title(name)
fig.colorbar(img, ax=axes.tolist(), shrink=.6, label="Cosine distance")
fig.subplots_adjust(left=.2, bottom=.42, right=.9, wspace=.7)
fig.savefig(OUT / f"{SELECTED_INPUT}-cross-word-distances.png", dpi=180, bbox_inches="tight")
plt.show()
for name, table in DISTANCE_TABLES.items():
    print(name); display(table.round(3))

## Optional shared UMAP

UMAP may expose nonlinear neighbourhood structure that two PCA axes miss. Fit once per
filter on the pooled training rows, with **no `y` argument**, then transform held-out rows.
The arithmetic mean of projected usages remains the century centroid, but UMAP distances
and especially centroid locations do not have the same interpretation as distances in the
original embedding space. Inspect several seeds before interpreting trajectory changes.

In [ ]:
if RUN_UMAP:
    import umap
    UMAP_COORDS = {}
    with threadpool_limits(limits=THREADS):
        for name in ["raw", "center_l2", "drop_pc_5"]:
            X = FILTERED[SELECTED_INPUT][name]
            model = umap.UMAP(n_components=2, metric="cosine", n_neighbors=30,
                             min_dist=.2, random_state=SEED, transform_seed=SEED, n_jobs=1)
            Z = np.empty((len(X), 2))
            Z[TRAIN] = model.fit_transform(X[TRAIN])
            Z[TEST] = model.transform(X[TEST])
            UMAP_COORDS[name] = Z
    plot_comparison(UMAP_COORDS, prefix=f"{SELECTED_INPUT}-umap")
else:
    print("Set RUN_UMAP=True for the nonlinear display comparison.")

## Interpretation and follow-up

- Compare each encoder separately with concatenation: scale and language coverage can
  affect whether concatenation helps classification yet hurts shared geometry.
- Compare filtering effects in the high-dimensional space with losses introduced by 2-D
  compression. A good high-dimensional filter may still have an inadequate 2-D PCA view.
- Reduced word predictability alone does not establish better semantics. Meaning and word
  identity are correlated; forcing them apart can erase the distinction we want to study.
- An MLP can retain useful signal even when kNN cannot read it. The MLP check is therefore
  stronger evidence of retention than scatter separation or silhouette alone.
- Repeat with different work splits, seeds and resampled century usages before selecting
  a method. Dates and work composition are part of the uncertainty; do not optimize for
  straight or smooth lines.
- A possible later experiment is learning directions predictive of a **specified nuisance**
  such as genre or author, then attenuating them. [INLP](https://aclanthology.org/2020.acl-main.647/)
  is a reference for this idea, not an implemented method here. Such removal can also
  erase correlated semantics or historical variation.

Outputs live in `figures/embedding-denoising/`; this notebook does not overwrite notebook
7's figures or classifier artifacts. It makes no claim that a particular PC is nuisance.